# **Remarriage After Divorce Proof of Concept**

#### **Team Members:** Khadiatou Ly & Zainab Atanda

##### Class: Machine Learning II (DS 4420)
##### Professor Gerber

In [1]:
# libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

/Users/kadely/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# loading files
fem = pd.read_csv("NSFG_2022_2023_FemRespPUFData.csv")
male = pd.read_csv("NSFG_2022_2023_MaleRespPUFData.csv")

print(fem.shape)
print(male.shape)

(5586, 1912)
(4371, 1157)


## **Cleaning**

We will synthesize the datasets and add a sex column. Once filtered for divorved individuals, we will further extract the data based on the socioeconomic factors used to conduct the analysis.

In [3]:
# extracting female and male divorcees
fem_div  = fem[fem["MARENDHX_1"] == 2].copy()
male_div = male[male["MARREND_1"] == 2].copy()

fem_div["remarried"]  = fem_div["WHMARHX_Y_2"].notna().astype(int)
male_div["remarried"] = male_div["MARDATEN_Y_2"].notna().astype(int)

print(f"\nFemale divorced: {len(fem_div)}  |  remarried: {fem_div['remarried'].sum()}")
print(f"Male   divorced: {len(male_div)}  |  remarried: {male_div['remarried'].sum()}")



Female divorced: 609  |  remarried: 306
Male   divorced: 37  |  remarried: 0


In [7]:
# pulling cols to keep
factors  = ["AGE_R", "POVERTY", "TOTINCR", "HIEDUC",
               "LABORFOR", "RSCRRACE", "HISP", "remarried"]

# changing col name to num_children
fem_div["num_children"]  = fem_div["NUMCHILD"]
male_div["num_children"] = male_div["NUMBIOKID"]


fem_div["sex"]  = 1
male_div["sex"] = 0

# combining to one common column list, then  into a single dataset
keep = factors + ["num_children", "sex"]
remarriage_data = pd.concat([fem_div[keep], male_div[keep]], ignore_index=True)

# standardizing col names
remarriage_data = remarriage_data.rename(columns={
    "AGE_R"    : "age",
    "POVERTY"  : "poverty_ratio",
    "TOTINCR"  : "total_income_cat",
    "HIEDUC"   : "education_level",
    "LABORFOR" : "labor_force_status",
    "RSCRRACE" : "race",
    "HISP"     : "hispanic",
})

print(f"\nCombined dataset: {remarriage_data.shape[0]} rows, {remarriage_data.shape[1]} columns")


Combined dataset: 646 rows, 10 columns


In [8]:
# replacing NFSG codes with NAN
remarriage_data["hispanic"] = remarriage_data["hispanic"].replace({8: np.nan, 9: np.nan})
remarriage_data["num_children"] = remarriage_data["num_children"].replace({98: np.nan, 99: np.nan})
remarriage_data["age"] = remarriage_data["age"].where(remarriage_data["age"] < 95, np.nan)
remarriage_data["poverty_ratio"] = remarriage_data["poverty_ratio"].where(remarriage_data["poverty_ratio"] <= 700, np.nan)

# recoding categorical variables

# hispanic - binary: 1 = Hispanic, 0 = not Hispanic
remarriage_data["hispanic"] = remarriage_data["hispanic"].map({1: 1, 5: 0})

# labor_force_status - binary: employed (codes 1–3) vs. not employed (codes 4–7)
remarriage_data["employed"] = remarriage_data["labor_force_status"].apply(
    lambda x: 1 if x in [1, 2, 3] else (0 if x in [4, 5, 6, 7] else np.nan)
)
remarriage_data = remarriage_data.drop(columns=["labor_force_status"])

# race - one-hot encode (White = reference category, dropped)
race_dummies = pd.get_dummies(remarriage_data["race"], drop_first=True, dummy_na=False)
race_dummies.columns = ["race_black", "race_aian", "race_asian_pi"]
remarriage_data = pd.concat([remarriage_data.drop(columns=["race"]), race_dummies.astype(float)], axis=1)

# imputing remaining NAN with median
for col in remarriage_data.columns:
    if remarriage_data[col].isna().any():
        remarriage_data[col] = remarriage_data[col].fillna(remarriage_data[col].median())

# check
assert remarriage_data.isna().sum().sum() == 0, "NO NaNs"
print("\nCleaning complete")
print(f"Remarried: {remarriage_data['remarried'].mean():.1%} | Not: {1 - remarriage_data['remarried'].mean():.1%}")
print(f"Final feature columns: {[c for c in remarriage_data.columns if c != 'remarried']}")


Cleaning complete
Remarried: 47.4% | Not: 52.6%
Final feature columns: ['age', 'poverty_ratio', 'total_income_cat', 'education_level', 'hispanic', 'num_children', 'sex', 'employed', 'race_black', 'race_aian', 'race_asian_pi']


## **Preprocessing for MLP**
We are splitting, testing, and training the data into their respective sets. Then we split into two class––remarried and not remarried

In [13]:
# features
features = ["age", "poverty_ratio", "total_income_cat", "education_level",
                "num_children", "hispanic", "sex", "employed",
                "race_black", "race_aian", "race_asian_pi"]

X = remarriage_data[features].values.astype(float)
y = remarriage_data["remarried"].values.astype(float)

In [14]:
# standardize parameters
def standardize(X_train, X_test):
    mu  = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return (X_train - mu) / std, (X_test - mu) / std

# train, test, and split
np.random.seed(42)
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

# splits into remarried and not remarried classes
def split_class(idx, frac=0.8):
    np.random.shuffle(idx)
    c = int(len(idx) * frac)
    return idx[:c], idx[c:]

pos_train, pos_test = split_class(pos_idx)
neg_train, neg_test = split_class(neg_idx)

train_idx = np.concatenate([pos_train, neg_train])
test_idx  = np.concatenate([pos_test,  neg_test])

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

X_train, X_test = standardize(X_train, X_test)
print(f"\nTraining set size: {len(X_train)}  |  Testing set size: {len(X_test)}")


Training set size: 516  |  Testing set size: 130


## **MLP**
We are now implementing an MLP manually.

- Architecture: Input → Hidden (32 neurons, ReLU) → Output (1 neuron, Sigmoid)
- Loss:         Binary Cross-Entropy
- Optimizer:    Mini-batch Stochastic Gradient Descent (SGD)